[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/en_clase/S11_01_cnn_en_clase.ipynb)

# S11 · Práctica 1 — Redes convolucionales (CNN): cómo "ve" una red · versión en clase

**Tiempo estimado:** 40 min en clase · **Sprint 11** (UD 4.4)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/codigo_explicado/S11_01_cnn.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** El cliente tiene miles de fotos de alojamientos, zonas comunes y platos, y hoy
nadie las usa. Antes de clasificarlas necesitas entender cómo una red **procesa una imagen**. Aquí lo verás
con imágenes pequeñas que se ejecutan sin internet.

**Al terminar sabrás:**
1. Qué es una imagen para un ordenador (matriz de números) y qué hace una **convolución** y un **pooling**.
2. Calcular a mano el tamaño de salida de una capa convolucional.
3. Construir, entrenar y evaluar una CNN pequeña con Keras y compararla con un MLP.
4. Mirar dentro de la red: filtros y mapas de activación; y usar *data augmentation*.

**Dos ramas.** (a) *Rama offline (todo el notebook)*: imágenes de dígitos 8×8 incluidas en scikit-learn.
(b) *Rama Colab (una celda marcada "requiere internet")*: descarga de `tf_flowers` para probar con fotos reales.

> 🏫 **Versión «en clase»** · ⏱ ≈ 40 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_01_cnn.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 1. Una imagen es una matriz de números

**Idea.** Una imagen en escala de grises es una tabla de píxeles con valores (aquí de 0 = blanco, sin tinta, a 16 = negro, tinta máxima).
Una imagen en color tiene **3 canales** (rojo, verde, azul): una tabla por color. Su forma es
`(alto, ancho, canales)`. Un lote de imágenes añade una dimensión delante: `(n_imágenes, alto, ancho, canales)`.

In [ ]:
digitos = load_digits()
imagenes, etiquetas = digitos.images, digitos.target      # (1797, 8, 8) y (1797,)
print("Forma del conjunto:", imagenes.shape, "· valores de", imagenes.min(), "a", imagenes.max())
print("Un dígito (matriz 8×8):\n", imagenes[0].astype(int))

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.4))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow(imagenes[i], cmap="gray_r"); eje.set_title(f"es un {etiquetas[i]}", fontsize=9); eje.axis("off")
plt.suptitle("Dígitos 8×8 de scikit-learn (miniaturas de 64 píxeles)"); plt.tight_layout(); plt.show()

**Ejercicio 1 · 🏫 en clase.** Prepara los datos para una red: en `X` guarda las imágenes **normalizadas entre 0 y 1**
(divide entre 16) con forma `(n, 8, 8, 1)` (Keras espera un canal explícito). Después crea con
`train_test_split` (`test_size=0.25`, `random_state=SEMILLA`, `stratify=etiquetas`) los conjuntos
`X_ent, X_pru, y_ent, y_pru`.

In [ ]:
# TODO: imagenes / 16.0 y añade el eje del canal con [..., np.newaxis]
...

assert X.shape == (1797, 8, 8, 1), "X debe tener forma (1797, 8, 8, 1): ¿has añadido el canal?"
assert X.max() <= 1.0 and X.min() >= 0.0, "Normaliza dividiendo entre 16 para que quede entre 0 y 1"
assert len(X_ent) + len(X_pru) == 1797 and len(X_pru) == 450, "Reparto 75 % / 25 %"

## 2. La convolución: un filtro que se desliza

**Idea.** Un **filtro** (o *kernel*) es una mini-tabla, por ejemplo 3×3. Se coloca sobre una zona de la imagen,
se multiplica casilla a casilla, se suma y se obtiene **un** número. Después se desliza una posición y se repite.
El resultado es un **mapa de activación**: dice *dónde* aparece el patrón que busca el filtro (un borde vertical,
una esquina…). Una CNN **aprende sus propios filtros** durante el entrenamiento.

Vamos a hacerlo a mano con NumPy sobre una imagen sencilla: un rectángulo claro en fondo oscuro.

In [ ]:
def convolucion2d(imagen, filtro):
    """Convolución 'válida' (sin relleno, paso 1) de una imagen 2D con un filtro 2D."""
    alto, ancho = imagen.shape
    k_alto, k_ancho = filtro.shape
    salida = np.zeros((alto - k_alto + 1, ancho - k_ancho + 1))
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            zona = imagen[i:i + k_alto, j:j + k_ancho]
            salida[i, j] = np.sum(zona * filtro)
    return salida

**Ejercicio 2 (calcular el tamaño de salida) · 🏫 en clase.** Para una entrada de `n` píxeles, filtro de `k`, relleno (*padding*) `p`
a cada lado y paso (*stride*) `s`, la salida mide `(n - k + 2p) / s + 1` (parte entera). Escribe
`tam_salida(n, k, p=0, s=1)` que devuelva ese número entero. Sirve para entender por qué las imágenes "encogen"
capa tras capa (y por qué Keras ofrece `padding="same"`).

In [ ]:
# TODO: Aplica la fórmula con división entera (//)
...

assert tam_salida(8, 3) == 6, "8 píxeles con filtro 3 sin relleno dan 6"
assert tam_salida(8, 3, p=1) == 8, "Con relleno 1 el tamaño se conserva ('same')"
assert tam_salida(224, 7, p=3, s=2) == 112, "Caso típico de las primeras capas de ResNet: 224 → 112"

In [ ]:
# Imagen de prueba: rectángulo claro sobre fondo oscuro
prueba = np.zeros((14, 14)); prueba[3:11, 4:10] = 1.0
filtro_vertical = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])     # detecta bordes verticales (Sobel)
filtro_horizontal = filtro_vertical.T                                  # detecta bordes horizontales

fig, ejes = plt.subplots(1, 3, figsize=(11, 3.6))
for eje, im, titulo in zip(ejes, [prueba, convolucion2d(prueba, filtro_vertical), convolucion2d(prueba, filtro_horizontal)],
                           ["Imagen original", "Bordes verticales", "Bordes horizontales"]):
    eje.imshow(im, cmap="gray"); eje.set_title(titulo); eje.axis("off")
plt.tight_layout(); plt.show()
print("Tamaño de la salida:", convolucion2d(prueba, filtro_vertical).shape, "(14 - 3 + 1 = 12)")

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_01_cnn.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
